In [ ]:
finite diff on a uniform
explain fornberg
BVP without fornberg
BVP with fornbeg
mesh choice improvements (chebyshev, lagrande)
compare all errors


# The Finite Difference Method we're used to  
Given an ordinary differential equation, we discretize it via a finite difference approximation of the derivative and represent it as a matrix problem.

Using the Taylor Series Expansion of u(x+h) and u(x-h)  
where we assume the matrix is uniform and where u(x) = $u_i$, u(x-h) = $u_{i-1}$, and u(x+h) = $u_{i+1}$  
### Finite Difference for approximations of derivatives

$u'(t) = \frac{u_{i+1} - u_{i-1}}{2h} + O(h^2)$  

$u''(t) = \frac{u_{i-1} +2u_i + u_{i+1}}{h^2} + O(h^2)$  

Then we proceed with the normal numerical scheme, however,  
what if we use a non-uniform grid where the distance between points isn't h  

### Non-uniform grid choice
Chebyshev nodes of the second kind:
$x_i = cos(\frac{i \pi}{N})$ for i=0,1,...,N  
If n = 10
$x_0 = 1.0$  
$x_1 = 0.95$  
$x_2 = 0.81$  
$x_3 = 0.58$  
...  

So now if we try to find the central finite difference at $x_2$ for example, we see the step size to the left is different than that to the right  
$h_1 = |x_2 - x_1 | =  0.14$  
$h_2 = | x_2 - x_3 | = 0.23$  
$u(x-h_1)$ and $u(x+h_2)$  
and hence instead of simply doing the taylor series expansion of $u(x+h) - u(x-h)$ to cancel out $u''$,  
it becomes a problem of solving a linear system to determine the weights  

### Fornberg Algorithm

### Example

### Use on a non-uniform Boundary Value Problem

In [17]:
# chebyshev nodes of second kind
a,b = -1,1
N=10
X = []
for i in 0:N
    x = cos(i*π/N)
    push!(X,x)
    println(x)
end


1.0
0.9510565162951535
0.8090169943749475
0.5877852522924731
0.30901699437494745
6.123233995736766e-17
-0.30901699437494734
-0.587785252292473
-0.8090169943749473
-0.9510565162951535
-1.0


In [ ]:
# fornberg i

# What happens when the grid isn't uniform?
a

In [ ]:
# Derivative tests for Fornberg (1998) Lagrange polynomial derivatives
# Bryan Kaiser
# 12/20/15

using DataArrays
using PyPlot
using PyCall
@pyimport numpy as np
@pyimport pylab as py


## ============================================================================
# function declaration

function weights(z,x,n,m)
# From Bengt Fornbergs (1998) SIAM Review paper.
#  	Input Parameters
#	z location where approximations are to be accurate,
#	x(0:nd) grid point locations, found in x(0:n)
#	n one less than total number of grid points; n must
#	not exceed the parameter nd below,
#	nd dimension of x- and c-arrays in calling program
#	x(0:nd) and c(0:nd,0:m), respectively,
#	m highest derivative for which weights are sought,
#	Output Parameter
#	c(0:nd,0:m) weights at grid locations x(0:n) for derivatives
#	of order 0:m, found in c(0:n,0:m)
#      	dimension x(0:nd),c(0:nd,0:m)

	c = zeros(n+1,m+1);
	c1 = 1.0;
	c4 = x[1+0]-z;
	for k=0:m
  		for j=0:n
    			c[1+j,1+k] = 0.0;
  		end
	end
	c[1+0,1+0] = 1.0;
	for  i=1:n
  		mn = min(i,m);
  		c2 = 1.0;
  		c5 = c4;
  		c4 = x[1+i]-z;
  		for j=0:i-1
    			c3 = x[1+i]-x[1+j];
    			c2 = c2*c3;
    			if (j == i-1)
      				for k=mn:-1:1
        				c[1+i,1+k] = c1*(k*c[1+i-1,1+k-1]-c5*c[1+i-1,1+k])/c2;
			      	end
      			c[1+i,1+0] = -c1*c5*c[1+i-1,1+0]/c2;
    			end
    			for k=mn:-1:1
      				c[1+j,1+k] = (c4*c[1+j,1+k]-k*c[1+j,1+k-1])/c3;
    			end
    			c[1+j,1+0] = c4*c[1+j,1+0]/c3;
  		end
  		c1 = c2;
	end
   	return c
end # weights function


## ============================================================================
# domain parameters

const Lx = 3000.0 # km, domain size
const Ly = Lx # km
const Lxcenter = 0.0 # x value @ the center of the grid
const Lycenter = 0.0 # y value @ the center of the grid
const N = 2^20 # series length (must be at least even)
const dx = Lx/float(N) # km, uniform longitudinal grid spacing
const dy = Ly/float(N) # km, uniform longitudinal grid spacing
x = collect(0.5*dx:dx:dx*N)-(Lx/2.0-Lxcenter) # km, centered uniform grid 


## ============================================================================
# signal

k = 2.0*pi/(Lx/5.0)
s = sin(x.*k)
dsdx = cos(x.*k).*k
d2sdx2 = -sin(x.*k).*k^2

figure(1)
plot(x,s,"b",x,dsdx,"r",x,d2sdx2,"g")
xlabel("x")
title("signal")
show()


##=============================================================================
# compute the uniform grid weights

c = weights(x[3],x[1:5],4,2)'; # 5 point stencil weights
dsF = zeros(1,N); # derivative of the input function
m = 1 # order of the derivative
   
dsF = zeros(Float64,N,1)
for j = 3:N-2      
	jm2 = j-2;
        jp2 = j+2;
        dsF[j,1] = vecdot(c[m+1,:],s[jm2:jp2]);
end
error = abs(dsdx-dsF)

@show(N)
figure(2)
semilogy(x[3:N-2],error[3:N-2],"k")
xlabel("x")
title("Lagrange polynomial 1st derivative error")
show()